Korzystanie z narzędzi generatywnej AI w rozwiązywaniu zadań nie jest dozwolone

<img src="no_AI.png" alt="Use of AI allowed only when properly documented " width="100" height="100">

# Zadanie obowiązkowe [0-10] pkt

Użyj zbioru danych [Yeast](https://archive.ics.uci.edu/dataset/110/yeast) i powtórz kroki z ćwiczeń.


1. [0-2 pkt] Używając [GridSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html), dokonaj przeszukania przestrzeni hiperparametrów kNN, zmieniając (uargumentuj wybór zakresów):
   1. liczbę sąsiadów
   1. wagę dla sąsiadów
   1. metrykę
1. [0-1.5 pkt] Do wyszukiwania dodaj miary skuteczności m.in. dokładność (*accuracy*) precyzję (*precision*), czułość (*recall*, *sensitivity*), czy też współczynnik korelacji Matthewsa (MCC). Gdzie to możliwe, dostosuj opcje miar, żeby uwzględniały niezbalansowanie zbioru
1. [0-0.5 pkt] Skomentuj wyniki uzyskane w punktach 1 i 2. Spróbuj zinterpretować wyniki
1. [0-2 pkt] Dokonaj selekcji cech, używając [SequentialFeatureSelector](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.SequentialFeatureSelector.html). Zmieniaj parametr `n_features_to_select` od dwóch do pięciu. Ile cech uzyskujemy przy opcji `auto`?
1. [0-0.5 pkt] Używając [Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html), sprawdź wpływ skalowania / normalizacji na wyniki (użyj m.in. `MinMaxScaler` oraz `RobustScaler`)
1. [0-1 pkt] Skomentuj wyniki uzyskane w punktach 4 i 5
1. [0-1.5 pkt] Dla najlepszej pary `n_features_to_select=2` wyryuj obszary decyzyjne. Skomentuj wyniki
1. [0-1 pkt] Czy w świetle uzyskanych wyników, kNN jest odpowiednim klasyfikatorem do tego problemu? Uzasadnij odpowiedź
  
<span style="color:red">**Uwaga:**</span> zadania bez komentarzy i wniosków zostaną ocenione na **0 punktów**.

# Realizacja zadania

### Załadowanie i wstępne przetwarzanie danych

(Zaczerpnięte z notebooka z ćwiczeń)

In [47]:
!pip install ucimlrepo

In [48]:
import pandas as pd
import numpy as np
from ucimlrepo import fetch_ucirepo

yeast = fetch_ucirepo(id=110)
X = yeast.data.features
y = yeast.data.targets

X.shape, len(y)

((1484, 8), 1484)

In [49]:
y.value_counts()

,count
localization_site,
CYT,463
NUC,429
MIT,244
ME3,163
ME2,51
ME1,44
EXC,35
VAC,30
POX,20


In [50]:
idx_to_drop = y[y['localization_site'] == 'ERL'].index
X = X.drop(idx_to_drop)
y = y.drop(idx_to_drop)

In [51]:
y.value_counts()

,count
localization_site,
CYT,463
NUC,429
MIT,244
ME3,163
ME2,51
ME1,44
EXC,35
VAC,30
POX,20


In [52]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

y_trans = label_encoder.fit_transform(y['localization_site'].values)

In [53]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

### Podział danych na zbiór treningowy i testowy

In [54]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_trans, stratify=y_trans,test_size=0.1,random_state=0)

### Trening i ewaluacja modelu z domyślnymi hiperparametrami

In [55]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score

knn = KNeighborsClassifier(n_jobs=-1)
knn.fit(X_train, y_train)

KNeighborsClassifier(n_jobs=-1)

In [56]:
y_pred_train = knn.predict(X_train)
y_pred_test = knn.predict(X_test)

In [57]:
print(f"{f1_score(y_train, y_pred_train, average='weighted'):.4f}")
print(f"{f1_score(y_test, y_pred_test, average='weighted'):.4f}")

0.6701
0.5387


In [58]:
print(f"{f1_score(y_train, y_pred_train, average='micro'):.4f}")
print(f"{f1_score(y_test, y_pred_test, average='micro'):.4f}")

0.6792
0.5541


In [59]:
print(f"{f1_score(y_train, y_pred_train, average='macro'):.4f}")
print(f"{f1_score(y_test, y_pred_test, average='macro'):.4f}")

0.6017
0.3848


In [60]:
from sklearn.metrics import matthews_corrcoef

print(f"{matthews_corrcoef(y_train, y_pred_train):.4f}")
print(f"{matthews_corrcoef(y_test, y_pred_test):.4f}")

0.5849
0.4148


In [61]:
knn.get_params()

{'algorithm': 'auto',
 'leaf_size': 30,
 'metric': 'minkowski',
 'metric_params': None,
 'n_jobs': -1,
 'n_neighbors': 5,
 'p': 2,
 'weights': 'uniform'}

## Przeszukiwanie przestrzeni hiperparametrów

In [62]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer, matthews_corrcoef

param_grid = {
    'n_neighbors': [3,5,7,9,11,15,21,31,45],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan', 'chebyshev']
}

scoring_metrics = {
    'accuracy': 'accuracy',
    'precision' : 'precision_weighted',
    'recall' : 'recall_macro',
    'mcc' : make_scorer(matthews_corrcoef),
    'f1' : 'f1_macro'
}

### Wybór hiperparametrów:
- n_neighbors - liczba sąsiadów - przeszukiwany jest szeroki zakres, od małych k, które mocno dopasowują się do danych treningowych, do dużych wartości, które dobrze uogólniają model, natomiast może wystąpić tu underfitting
- weights - wagi które określają wpływ sąsiadów na ostateczne przydzielenie próbki do klasy. Uniform przydziela równe wagi dla każdego sąsiada, natomiast distance określa wagę sąsiada odwrotnie proporcjonalnie do jego odległości od próbki.
- metric - metryki od p=1 manhattan, p=2 euclidean, oraz chebyshev dla p->inf. Zamiast tak określonych metryk można użyć metryki uogólnionej - minkowski - i manipulować wartością parametru p. Minkowski z p=1 to metryka manhattan, z p=2 (domyślnie w KNNeighborClassifier) to euklidean, itd.

In [63]:
knn_base = KNeighborsClassifier(n_jobs=-1)

grid_search = GridSearchCV(
    estimator=knn_base,
    param_grid=param_grid,
    cv=5,
    scoring=scoring_metrics,
    refit='mcc',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 54 candidates, totalling 270 fits


GridSearchCV(cv=5, estimator=KNeighborsClassifier(n_jobs=-1), n_jobs=-1,
             param_grid={'metric': ['euclidean', 'manhattan', 'chebyshev'],
                         'n_neighbors': [3, 5, 7, 9, 11, 15, 21, 31, 45],
                         'weights': ['uniform', 'distance']},
             refit='mcc',
             scoring={'accuracy': 'accuracy', 'f1': 'f1_macro',
                      'mcc': make_scorer(matthews_corrcoef, response_method='predict'),
                      'precision': 'precision_weighted',
                      'recall': 'recall_macro'},
             verbose=1)

In [64]:
results_df = pd.DataFrame(grid_search.cv_results_)
best_idx = grid_search.best_index_
print("Wyniki miar dla najlepszego (według mcc) modelu:")
print(f"Accuracy: {results_df.loc[best_idx, 'mean_test_accuracy']:.4f}")
print(f"Precision: {results_df.loc[best_idx, 'mean_test_precision']:.4f}")
print(f"Recall: {results_df.loc[best_idx, 'mean_test_recall']:.4f}")
print(f"MCC: {results_df.loc[best_idx, 'mean_test_mcc']:.4f}")
print(f"F1: {results_df.loc[best_idx, 'mean_test_f1']:.4f}")

Wyniki miar dla najlepszego (według mcc) modelu:
Accuracy: 0.6145
Precision: 0.6121
Recall: 0.5511
MCC: 0.4991
F1: 0.5507


In [65]:
grid_search.best_params_

{'metric': 'euclidean', 'n_neighbors': 15, 'weights': 'distance'}

## Komentarz i interpretacja wyników

Zoptymalizowanie hiperparametrów za pomoca gridsearch przyniosło znaczną poprawę wyników modelu.
Zarówno dla metryki f1 z parametrem average=macro (najlepszym dla naszego niezbalansowanego zbioru), gdzie wynik dla danych testowych podniósł się z 0.3848 dla domyślnych parametrów, do 0.5507 dla parametrów zoptymalizowanych.
Również dla metryki mcc wynik został poprawiony - z 0.4148 do 0.4991. Metryka mcc (Matthews Correlation Coefficient) uwzględnia wszystkie 4 wartości z macierzy pomyłek i dobrze sprawdza się w niezbalansowanych zbiorach. Metryka przyjmuje wartości między -1 a 1, gdzie 0 to model zupełnie losowy, więc można zauważyć, że nasz model radzi sobie znacznie lepiej, niż gdyby klasa zostałaby przydzielana zupełnie losowo.

Model ma najlepsze wyniki dla względnie wysokiego k=15 (domyślnie k=5), co zapobiega overfittingowi i uodparnia model na lokalny szum.
Dla hiperparametru weights, najlepsze wyniki uzyskał model wykorzystujący distace. Dla względnie dużego k, mniejsze klasy mogłyby zostać przeważone przez klasy liczniejsze, gdyby wykorzystany został uniform.
Domyślna metryka euclidean (czyli tak naprawdę minkowski dla p=2) dała lepszy rezultat niż pozostałe 2 przedstawione w ramach laboratorium.

Dla niezbalansowanego zbioru z jakim mamy tu doczynienia, najbardziej miarodajne są metryki f1 z argumentem average=macro oraz mcc. Analizując wynik tych metryk, model daje w miarę zadowalające wyniki, uwzględniając, że zbiór jest wysoce niezbalansowany oraz przewidujemy 9 rozróżnialnych klas.